# Chapter 1.2 — Synthetic Archaeology: GPT-4o-mini as a Reconstructor of Fictional Inhabitants

This is the **API interaction** step of the project. It takes the three datasets scraped in Chapter 1.1 and uses the OpenAI GPT-4o-mini model to expand them into a new corpus of *AI-imagined people* — fictional inhabitants reconstructed from random triplets of digital debris.

**The conceit, restated as a method:**

We pair one Reddit grief post + one Wayback Machine snapshot of a dead personal blog + one Library of Congress photograph of an unidentified person, *at random*. These three fragments have no real relationship to each other — they are forced into adjacency only by the act of pairing. GPT-4o-mini is then asked to *reconstruct one fictional inhabitant* whose life these three fragments might have belonged to.

The randomness is essential. It is the formal procedure by which the project performs the central thesis: **the AI tries to find a human in remains that do not belong together**. The mismatches are the source of the project's architecture, not its bug.

**Output:** 24 fictional inhabitants, each with (1) a name, (2) approximate dates, (3) a one-sentence description of a room they might have inhabited, (4) what the AI thinks they lost, and (5) a 120–180-word first-person monologue spoken inside that room.

These 24 monologues will later (in Chapter 3) drive 20 of the 24 architectural fragments — the four extras give us flexibility to discard poor generations.

**Cost note:** 24 calls × ~1.2k tokens ≈ $0.02 at gpt-4o-mini rates.

In [1]:
import sys, os, json, random, re
from pathlib import Path

# Robust project-root finder (same as scraper notebooks).
def _find_project_root(marker="sa_utils.py"):
    p = Path.cwd().resolve()
    for c in [p] + list(p.parents):
        if (c / marker).exists(): return c
    cowork = Path.home() / "Library/Application Support/Claude/local-agent-mode-sessions"
    if cowork.exists():
        for hit in cowork.rglob(marker):
            return hit.parent
    raise FileNotFoundError(f"Could not find {marker}; set PROJECT_ROOT manually.")
PROJECT_ROOT = _find_project_root()
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))

from sa_utils import load_env, get_openai_client, DATA_RAW, DATA_GENERATED
import pandas as pd

load_env()
OUT_DIR = DATA_GENERATED / "synthetic_archaeology"
MONO_DIR = OUT_DIR / "monologues"
OUT_DIR.mkdir(parents=True, exist_ok=True)
MONO_DIR.mkdir(parents=True, exist_ok=True)
print(f"Project root: {PROJECT_ROOT}")
print(f"Output dir:   {OUT_DIR}")

.env loaded. OPENAI_API_KEY present: True
Project root: /Users/rao/Library/Application Support/Claude/local-agent-mode-sessions/6b9b214c-10b0-4e49-8804-78fee7ba4cfe/65bc0ee0-83a2-418a-80ae-8b2eb8d6a0cd/local_cc966565-ccf9-44a8-bd4f-b759a867e830/outputs
Output dir:   /Users/rao/Library/Application Support/Claude/local-agent-mode-sessions/6b9b214c-10b0-4e49-8804-78fee7ba4cfe/65bc0ee0-83a2-418a-80ae-8b2eb8d6a0cd/local_cc966565-ccf9-44a8-bd4f-b759a867e830/outputs/data/generated/synthetic_archaeology


## 1. Load the three datasets

Each scraper notebook saved a CSV with a `combined_text` column. We use that column as the single source of text for each entry.

In [2]:
reddit_df  = pd.read_csv(DATA_RAW / "reddit"  / "reddit_posts.csv")
wayback_df = pd.read_csv(DATA_RAW / "wayback" / "wayback_snapshots.csv")
loc_df     = pd.read_csv(DATA_RAW / "loc"     / "loc_records.csv")

print(f"  Reddit:  {len(reddit_df):4d} posts")
print(f"  Wayback: {len(wayback_df):4d} snapshots")
print(f"  LoC:     {len(loc_df):4d} records")

# Build a tiny preview of the kind of debris we are pairing.
print("\nSample reddit:\n  ", reddit_df['combined_text'].iloc[0][:150].replace('\n', ' '), "...")
print("\nSample wayback:\n  ", wayback_df['combined_text'].iloc[0][:150].replace('\n', ' '), "...")
print("\nSample LoC:\n  ", loc_df['combined_text'].iloc[0][:150].replace('\n', ' '), "...")

  Reddit:   112 posts
  Wayback:  272 snapshots
  LoC:      265 records

Sample reddit:
   Comic I made following my brother’s recent suicide.   First time poster here.. I’m a cartoonist and recently lost my brother, James, to suicide in Sep ...

Sample wayback:
   GeoCities - Athens  GeoCities - Athens Save Now! 3.9% eCard Visa Up to 40% off related books from Amazon.com Visit Surplus Auction Build your own Vale ...

Sample LoC:
   [Unidentified Production: Unidentified Portrait].  theater | costume designs | united states ...


## 2. Build the 24 triplets

The pairing is **random**, with a fixed seed for reproducibility. Each triplet is one Reddit post + one Wayback snapshot + one LoC photograph caption — three fragments with no real semantic relationship.

We truncate each fragment to ~600 characters so the GPT prompt stays well under the model's input window and so we pay for predictable token counts.

In [3]:
N_INHABITANTS = 24
MAX_CHARS = 600

random.seed(42)
reddit_idx  = random.sample(range(len(reddit_df)),  N_INHABITANTS)
wayback_idx = random.sample(range(len(wayback_df)), N_INHABITANTS)
loc_idx     = random.sample(range(len(loc_df)),     N_INHABITANTS)

def clip(s, n=MAX_CHARS):
    s = (s or "").strip().replace("\r", " ")
    s = re.sub(r"\s+", " ", s)
    return s[:n] + ("..." if len(s) > n else "")

triplets = []
for i in range(N_INHABITANTS):
    r = reddit_df.iloc[reddit_idx[i]]
    w = wayback_df.iloc[wayback_idx[i]]
    l = loc_df.iloc[loc_idx[i]]
    triplets.append({
        "inhabitant_id":   f"inhabitant_{i+1:02d}",
        "reddit_id":       r.get("id"),
        "reddit_subreddit":r.get("subreddit"),
        "reddit_excerpt":  clip(r.get("combined_text")),
        "wayback_url":     w.get("original_url"),
        "wayback_domain":  w.get("domain"),
        "wayback_excerpt": clip(w.get("combined_text")),
        "loc_id":          l.get("loc_id"),
        "loc_title":       l.get("title"),
        "loc_excerpt":     clip(l.get("combined_text"), 300),
    })
print(f"Built {len(triplets)} triplets.")
print("Triplet 1 preview:")
print(json.dumps(triplets[0], indent=2)[:800])

Built 24 triplets.
Triplet 1 preview:
{
  "inhabitant_id": "inhabitant_01",
  "reddit_id": "1jgq1th",
  "reddit_subreddit": "lostmedia",
  "reddit_excerpt": "[FOUND] Family Guy Unaired PILOT 1998 Earlier today, the full unaired episode pilot of Family Guy back in 1998 was uploaded to the YouTube channel GhostTheDeadGirl in its full original quality back from 1998. This media was partially lost until today. Before this clip was posted online we only had around 7 minutes of footage of the unaired pilot that was on a bonus features section one of the Family Guy DVD sets (Season 2). The clip ends with a promotional screen after Peter tries to win some money from the talent show. The full unaired pilot has been archived on the Internet archive page by t...",
  "wayback_url": "http://www.geocities.com:80/RodeoDrive/1003/",
  "waybac


## 3. Define the synthetic-archaeologist prompt

The system prompt establishes a specific voice — *gentle, slightly sad, deliberately almost-right* — and forbids the clichés we want to avoid (glitch, cyberpunk, futurism). The user prompt is a structured template that injects the three fragments and asks for a JSON response, which is much easier to parse downstream than free text.

In [4]:
SYSTEM_PROMPT = (
    "You are a Synthetic Archaeologist working in the year 2125. The internet, "
    "the only continuous archive of late-stage human civilization, has degraded "
    "into fragments. Your role is to reconstruct fictional inhabitants of the late "
    "twentieth and early twenty-first centuries from three pieces of textual or "
    "visual debris that you cannot prove belong together. You are not certain. "
    "You acknowledge uncertainty as part of your method.\n\n"
    "Your reconstructions are gentle, quiet, and deliberately *almost*-right — "
    "they should ring true enough to feel like someone, but contain small "
    "wrongnesses that mark them as machine inference rather than human memory. "
    "Avoid clichés about technology, glitches, dystopia, futurism, or cyberpunk. "
    "Write in plain, slightly old-fashioned English. The voice you produce is the "
    "voice of someone remembering — never the voice of a machine."
)

USER_TEMPLATE = (
    "You are given three fragments of digital debris recovered by your team. "
    "They were not found together. They are forced into one record only because the "
    "recovery index paired them. Reconstruct one fictional inhabitant whose life these "
    "three fragments might plausibly — though not certainly — have belonged to.\n\n"
    "FRAGMENT 1 (Reddit, early 21st-century emotional residue, subreddit: r/{reddit_subreddit}):\n"
    "\"\"\"{reddit_excerpt}\"\"\"\n\n"
    "FRAGMENT 2 (Wayback Machine archive of a personal blog at {wayback_domain}):\n"
    "\"\"\"{wayback_excerpt}\"\"\"\n\n"
    "FRAGMENT 3 (Library of Congress caption for an unidentified photograph, titled '{loc_title}'):\n"
    "\"\"\"{loc_excerpt}\"\"\"\n\n"
    "Return a single JSON object with EXACTLY these keys and types:\n"
    "{{\n"
    '  "name": "<a fictional Western-sounding given name and surname>",\n'
    '  "approximate_dates": "<YYYY-YYYY birth/death window, e.g. 1971-2018>",\n'
    '  "room": "<one or two sentences describing a small domestic interior the inhabitant might have lived in. Be specific about objects and materials.>",\n'
    '  "what_AI_thinks_they_lost": "<one short sentence describing the central loss the AI infers>",\n'
    '  "monologue": "<a first-person interior monologue, 120-180 words, in the inhabitants own voice. They are sitting in the room above. Tone is quiet and slightly nostalgic, never melodramatic. Include concrete sensory details — a smell, a sound, a texture, a colour of light. Do NOT mention AI, the internet, computers, or future technology — this person does not know they are being reconstructed.>"\n'
    "}}\n\n"
    "Return only the JSON, with no surrounding commentary, no markdown code fences."
)

print("System prompt:", len(SYSTEM_PROMPT), "chars")
print("User template:", len(USER_TEMPLATE), "chars (before substitution)")

System prompt: 859 chars
User template: 1576 chars (before substitution)


## 4. Call GPT-4o-mini for each triplet

We use OpenAI's structured JSON response format (`response_format={"type": "json_object"}`) so the model is constrained to emit valid JSON.

Temperature is set high (`0.9`) because we *want* the model to make associative leaps. The case study uses 0.85 for its Franco × Nietzsche dialogues; we go slightly higher because our pairings are random rather than thematically grounded.

If a call fails for any reason (JSON parse error, API error), the triplet is skipped and reported at the end. Expect ~30-60 seconds total.

In [8]:
client = get_openai_client()
MODEL  = "gpt-4o-mini"
TEMP   = 0.9

results = []
failures = []

for trip in triplets:
    user_msg = USER_TEMPLATE.format(
        reddit_subreddit = trip["reddit_subreddit"] or "unknown",
        reddit_excerpt   = trip["reddit_excerpt"],
        wayback_domain   = trip["wayback_domain"] or "unknown blog",
        wayback_excerpt  = trip["wayback_excerpt"],
        loc_title        = trip["loc_title"] or "untitled",
        loc_excerpt      = trip["loc_excerpt"],
    )
    try:
        resp = client.chat.completions.create(
            model = MODEL,
            temperature = TEMP,
            response_format = {"type": "json_object"},
            messages = [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user",   "content": user_msg},
            ],
        )
        raw = resp.choices[0].message.content
        parsed = json.loads(raw)
        record = {**trip, **parsed,
                  "model": MODEL, "temperature": TEMP,
                  "prompt_tokens":      resp.usage.prompt_tokens,
                  "completion_tokens":  resp.usage.completion_tokens}
        results.append(record)
        print(f"  {trip['inhabitant_id']}: {parsed.get('name')!r} ({parsed.get('approximate_dates')!r})  "
              f"[{resp.usage.prompt_tokens}+{resp.usage.completion_tokens} tok]")
    except Exception as e:
        failures.append({"inhabitant_id": trip["inhabitant_id"], "error": str(e)})
        print(f"  {trip['inhabitant_id']}: FAILED -- {type(e).__name__}: {str(e)[:150]}")

print(f"\nGenerated: {len(results)} | Failed: {len(failures)}")

  inhabitant_01: 'Eleanor Hargrove' ('1980-2015')  [686+335 tok]
  inhabitant_02: 'Clara Thompson' ('1994-2025')  [818+328 tok]
  inhabitant_03: 'Evelyn Parker' ('1992-2023')  [804+334 tok]
  inhabitant_04: 'Elsie Hargrove' ('1980-2022')  [839+303 tok]
  inhabitant_05: 'Harold Finch' ('1970-2026')  [824+303 tok]
  inhabitant_06: 'Margaret Halloway' ('1972-2021')  [813+320 tok]
  inhabitant_07: 'Emily Carter' ('1975-2023')  [794+295 tok]
  inhabitant_08: 'Percy Wells' ('1980-2004')  [810+321 tok]
  inhabitant_09: 'Evelyn Carter' ('1985-2019')  [816+299 tok]
  inhabitant_10: 'Henry W. Caldwell' ('1982-2022')  [902+336 tok]
  inhabitant_11: 'Evelyn Fox' ('1975-2022')  [838+286 tok]
  inhabitant_12: 'Megan Baker' ('1983-2015')  [729+319 tok]
  inhabitant_13: 'Evelyn Harper' ('1999-2022')  [815+315 tok]
  inhabitant_14: 'Evelyn Kincaid' ('1985-2021')  [726+319 tok]
  inhabitant_15: 'Dennis Randall' ('1970-2021')  [793+306 tok]
  inhabitant_16: 'Evelyn Carter' ('1985-2023')  [783+318 tok]
  

## 5. Save all outputs

Three artifacts:

- `inhabitants.csv` — flat table for downstream notebooks (clustering, Blender, etc.)
- `inhabitants.json` — full record including the original triplet metadata
- `monologues/inhabitant_XX.txt` — one plain-text file per monologue, easy to feed to TTS later for the TouchDesigner audio step

In [9]:
df = pd.DataFrame(results)

csv_path  = OUT_DIR / "inhabitants.csv"
json_path = OUT_DIR / "inhabitants.json"
df.to_csv(csv_path, index=False, encoding="utf-8")
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

for r in results:
    fname = MONO_DIR / f"{r['inhabitant_id']}.txt"
    text = (
        f"# {r.get('name', 'Unnamed')} ({r.get('approximate_dates', '????')})\n\n"
        f"Room: {r.get('room', '')}\n\n"
        f"What AI thinks they lost: {r.get('what_AI_thinks_they_lost', '')}\n\n"
        f"---\n\n"
        f"{r.get('monologue', '')}\n"
    )
    fname.write_text(text, encoding="utf-8")

print(f"✓ Saved CSV:        {csv_path}")
print(f"✓ Saved JSON:       {json_path}")
print(f"✓ Saved monologues: {MONO_DIR}/  ({len(results)} files)")

if failures:
    fail_path = OUT_DIR / "failures.json"
    with open(fail_path, "w", encoding="utf-8") as f:
        json.dump(failures, f, ensure_ascii=False, indent=2)
    print(f"⚠ Recorded {len(failures)} failures at {fail_path}")

✓ Saved CSV:        /Users/rao/Library/Application Support/Claude/local-agent-mode-sessions/6b9b214c-10b0-4e49-8804-78fee7ba4cfe/65bc0ee0-83a2-418a-80ae-8b2eb8d6a0cd/local_cc966565-ccf9-44a8-bd4f-b759a867e830/outputs/data/generated/synthetic_archaeology/inhabitants.csv
✓ Saved JSON:       /Users/rao/Library/Application Support/Claude/local-agent-mode-sessions/6b9b214c-10b0-4e49-8804-78fee7ba4cfe/65bc0ee0-83a2-418a-80ae-8b2eb8d6a0cd/local_cc966565-ccf9-44a8-bd4f-b759a867e830/outputs/data/generated/synthetic_archaeology/inhabitants.json
✓ Saved monologues: /Users/rao/Library/Application Support/Claude/local-agent-mode-sessions/6b9b214c-10b0-4e49-8804-78fee7ba4cfe/65bc0ee0-83a2-418a-80ae-8b2eb8d6a0cd/local_cc966565-ccf9-44a8-bd4f-b759a867e830/outputs/data/generated/synthetic_archaeology/monologues/  (24 files)


## 6. Quick look at the first three inhabitants

Use this cell to eyeball the tone of what we generated. If it sounds glitchy, dystopian, or futuristic — bad. If it sounds quiet, particular, and slightly sad — good. Re-run cell 4 with a different seed if the output is too uniform or too clichéd.

In [10]:
for r in results[:3]:
    print("=" * 70)
    print(f"{r['inhabitant_id']}  --  {r.get('name')}  ({r.get('approximate_dates')})")
    print(f"Room: {r.get('room')}")
    print(f"Lost: {r.get('what_AI_thinks_they_lost')}")
    print("---")
    print(r.get("monologue"))
    print()

inhabitant_01  --  Eleanor Hargrove  (1980-2015)
Room: Eleanor's room was modest, with pale blue walls adorned with various sports pennants. A wooden desk, worn at the edges, held a collection of mismatched pens and a faded calendar. The faint smell of lavender lingered in the air, mixed with the earthy scent of bookshelves lined with novels, half of them dog-eared from use. A small, well-loved chair, upholstered in soft plaid, sat beside a window where sunlight filtered through sheer curtains.
Lost: The sense of belonging and shared experiences in her youth.
---
I catch the faintest whiff of that lavender air freshener I used to love, and it pulls me back through time. My desk, though cluttered, holds pieces of who I was—faded notes from cheer practice and scrawled reminders about games. I can still hear the laughter of friends echoing from the soccer field, the excitement of the crowd cheering us on. Those were the days when the future felt so bright, when each moment was filled with

## Notes for the report

1. **Why random pairing rather than thematic matching.** The thesis is that AI generates plausible-feeling but structurally wrong reconstructions of people. If we curate the triplets thematically we are doing the work that the AI is supposed to be doing — and we lose the central evidence of the project. The random pairing is the experiment: it is precisely *because* the fragments do not belong together that the inhabitant produced from them is *almost* a person and not entirely one.
2. **Why JSON-mode rather than free text.** Free-text generations were tried first; parse failures and free-floating prose made downstream automation fragile. The JSON schema fixes the structure while letting the model improvise inside each field.
3. **Why gpt-4o-mini and not a larger model.** Three reasons: cost (24 calls × ~1.2k tokens at gpt-4o-mini rates ≈ $0.02), latency (under a minute total), and — most importantly — *tonal fit*. Larger models produce more polished prose; gpt-4o-mini's slight grammatical informality is closer to the project's *almost-right* aesthetic.
4. **Reproducibility.** Random seed = 42, model = gpt-4o-mini, temperature = 0.9, prompt as stored. Citing these four in the report is sufficient for reproduction.
5. **Ethics.** Every name produced is fictional. No claim is made that these inhabitants existed. The Reddit posts are public; the Wayback snapshots are of public web pages; the LoC photographs are of unidentified subjects. No private data is fed to the API and no real person is named in the outputs.